# Scraper masivo de Importaciones SUNAT (Python)

Traducción a Python del script en R que consulta y descarga el detalle de
importaciones por subpartida nacional en el portal de Aduanas SUNAT
(aduanet.gob.pe), para **una lista de subpartidas x un rango de años**.

Toda la lógica de scraping vive en `src/sunat_scraper.py` (compartida con
el dashboard `dashboard.py`), así que este notebook es solo orquestación:
arma la lista de combinaciones subpartida x año y llama a
`procesar_subpartida_anio` para cada una.

Por defecto trae como ejemplo la lista de subpartidas de vehículos
(capítulos 87.03/87.04/87.11) usada en el script original, pero
`CANDIDATOS_SUBPARTIDA` se puede reemplazar por cualquier lista de códigos
de 10 dígitos (por ejemplo, tomados de `data/subpartidas_completo.csv`).

**Notas de funcionamiento del portal** (descubiertas por prueba y error):
- El período de consulta (`fini`/`ffin`) debe estar **dentro del mismo año**.
  No se puede pedir un rango que cruce de un año a otro en una sola consulta.
- SUNAT procesa cada requerimiento de forma asíncrona: hay que hacer
  *polling* de la tabla de resultados hasta que el reporte quede listo.

## 0. Configuración general

In [ ]:
import sys
import time
from pathlib import Path
from datetime import date

import pandas as pd

# El módulo con la lógica de scraping vive en src/, un nivel arriba de notebooks/
RAIZ = Path.cwd().parent if (Path.cwd() / ".." / "src").exists() else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from sunat_scraper import (
    nueva_sesion,
    validar_subpartida,
    procesar_subpartida_anio,
    ESTADO_COMPLETADO,
)

In [ ]:
# --- Carpeta de trabajo (datos crudos y consolidado quedan en data/) ---
DIR_DATA = RAIZ / "data"
CARPETA_RESULTADOS = DIR_DATA / "resultados_sunat"
CARPETA_DBF = DIR_DATA / "dbf_extraidos"
for carpeta in (DIR_DATA, CARPETA_RESULTADOS, CARPETA_DBF):
    carpeta.mkdir(parents=True, exist_ok=True)

# --- Rango de años ---
ANIO_INICIO = 2007
ANIO_FIN = date.today().year
MES_FIN_ULTIMO_ANIO = date.today().month
DIA_FIN_ULTIMO_ANIO = date.today().day

# --- Tipo de régimen: 'Impo' = importaciones, 'Expo' = exportaciones ---
REGIMEN = "Impo"

# --- Pausas y polling ---
PAUSA_VALIDACION = 2
RENOVAR_SESION_CADA = 20        # combinaciones entre cada renovación de sesión
MAX_ESPERA_SEG = 1800           # máximo a esperar a que SUNAT genere un reporte
INTERVALO_POLL_SEG = 25         # cada cuánto se re-consulta el estado

## 1. Subpartidas candidatas

Ejemplo: unión de todos los códigos que aparecieron en los Aranceles de
Aduanas oficiales de 2002, 2007, 2012, 2017 y 2022 para los capítulos 87.03
(automóviles), 87.04 (camiones) y 87.11 (motocicletas).

**Reemplaza esta lista** por los códigos que te interesen -- por ejemplo,
cargando `data/subpartidas_completo.csv` y filtrando por capítulo o
descripción. Los que ya no existen se descartan solos en el paso de
validación.

In [ ]:
CANDIDATOS_SUBPARTIDA = [
    "8703100000",
    "8703210010", "8703210090",
    "8703220010", "8703220020", "8703221000", "8703229010", "8703229020", "8703229090",
    "8703230010", "8703230090", "8703231000", "8703239010", "8703239020", "8703239090",
    "8703240020", "8703240090", "8703241000", "8703249010", "8703249020", "8703249090",
    "8703310010", "8703310090", "8703311000", "8703319010", "8703319020", "8703319090",
    "8703320010", "8703320020", "8703320090", "8703321000", "8703329010", "8703329020", "8703329090",
    "8703330010", "8703330020", "8703330090", "8703331000", "8703339010", "8703339020", "8703339090",
    "8703401000", "8703409010", "8703409020", "8703409090",
    "8703501000", "8703509010", "8703509020", "8703509090",
    "8703601000", "8703609010", "8703609020", "8703609090",
    "8703701000", "8703709010", "8703709020", "8703709090",
    "8703801000", "8703809010", "8703809020", "8703809090",
    "8703900010", "8703900020", "8703900090",
    "8704100000",
    "8704210090", "8704211010", "8704211090", "8704219000",
    "8704220000", "8704221000", "8704222000", "8704229000",
    "8704230000",
    "8704310010", "8704310090", "8704311010", "8704311090", "8704319000",
    "8704320000", "8704321000", "8704322000", "8704329000",
    "8704411000", "8704419000", "8704420000", "8704430000",
    "8704511000", "8704519000", "8704520000",
    "8704601000", "8704609000",
    "8704900000", "8704901000", "8704901100", "8704901900",
    "8704902100", "8704902900", "8704903100", "8704903900",
    "8704904100", "8704904900", "8704905100", "8704905900",
    "8704909000", "8704909100", "8704909900",
    "8711100000", "8711200000", "8711300000", "8711400000", "8711500000",
    "8711600000", "8711600010", "8711600090", "8711900000",
]

In [ ]:
sesion = nueva_sesion()
print(f"Validando {len(CANDIDATOS_SUBPARTIDA)} subpartidas candidatas...")

resultados_validacion = []
for sp in CANDIDATOS_SUBPARTIDA:
    resultados_validacion.append(validar_subpartida(sp, sesion))
    time.sleep(PAUSA_VALIDACION)

validacion_df = pd.DataFrame(resultados_validacion)
validacion_df.to_csv(DIR_DATA / "validacion_subpartidas.csv", index=False)

subpartidas = validacion_df.loc[validacion_df["valida"] == True, "subpartida"].tolist()
if not subpartidas:
    raise RuntimeError("Ninguna subpartida candidata fue válida. Revisa 'validacion_subpartidas.csv'.")
print(f"{len(subpartidas)} de {len(CANDIDATOS_SUBPARTIDA)} subpartidas son válidas.")

## 2. Armar combinaciones subpartida x año

In [ ]:
combinaciones = []
for anio in range(ANIO_INICIO, ANIO_FIN + 1):
    for sp in subpartidas:
        combinaciones.append({"subpartida": sp, "anio": anio})

print(f"{len(combinaciones)} combinaciones (subpartida x año, {ANIO_INICIO}-{ANIO_FIN}) por procesar.")

## 3. Procesar cada combinación de punta a punta

Por cada combinación: envía la consulta, hace *polling* hasta que SUNAT
genere el reporte, descarga el `.ZIP`, lo descomprime y lee el `.DBF` --
todo encapsulado en `procesar_subpartida_anio` (`src/sunat_scraper.py`).

Esto puede tardar bastante (SUNAT procesa de forma asíncrona); el progreso
se va imprimiendo por combinación.

In [ ]:
def imprimir_estado(info):
    print(f"  [{info['subpartida']} / {info['anio']}] {info['estado']}: {info['mensaje']}")


resumen = []
dataframes = []
sesion = nueva_sesion()

for i, c in enumerate(combinaciones, start=1):
    if i % RENOVAR_SESION_CADA == 1:
        sesion = nueva_sesion()

    print(f"[{i}/{len(combinaciones)}] {c['subpartida']} - {c['anio']}")

    kwargs = dict(
        regi=REGIMEN,
        max_espera_seg=MAX_ESPERA_SEG,
        intervalo_poll_seg=INTERVALO_POLL_SEG,
        on_status=imprimir_estado,
    )
    if c["anio"] == ANIO_FIN:
        kwargs.update(mes_fin=MES_FIN_ULTIMO_ANIO, dia_fin=DIA_FIN_ULTIMO_ANIO)

    resultado = procesar_subpartida_anio(
        c["subpartida"], c["anio"], sesion, CARPETA_RESULTADOS, CARPETA_DBF, **kwargs
    )

    resumen.append({
        "subpartida": resultado.subpartida, "anio": resultado.anio,
        "estado": resultado.estado, "mensaje": resultado.mensaje,
        "registros": resultado.registros,
    })
    if resultado.datos is not None:
        dataframes.append(resultado.datos)

    # Guardar resumen incremental por si el notebook se corta a medio camino
    pd.DataFrame(resumen).to_csv(DIR_DATA / "resumen_descarga_sunat.csv", index=False)

resumen_df = pd.DataFrame(resumen)
n_ok = (resumen_df["estado"] == ESTADO_COMPLETADO).sum()
print(f"\nCompletadas con datos: {n_ok}/{len(resumen_df)}")

## 4. Consolidar todo en un único CSV

In [ ]:
if dataframes:
    datos_completos = pd.concat(dataframes, ignore_index=True)
    salida = DIR_DATA / "importaciones_consolidado.csv"
    datos_completos.to_csv(salida, index=False, encoding="utf-8")
    print(f"Consolidado final: {len(datos_completos)} filas en '{salida}'.")
else:
    print("No se descargó ningún dato -- revisa 'resumen_descarga_sunat.csv'.")